# 6 — Backtracking : choisir, explorer, annuler


## Fil conducteur

La récursion du chapitre 5 suivait une réduction déterminée. Ici, un état peut
offrir plusieurs choix. Nous allons donc former un arbre de décisions : choisir,
explorer, puis restaurer exactement l'état avant d'essayer le choix suivant.




*Chaque niveau ajoute un choix ; revenir au parent permet d’explorer la branche suivante.*


## 1. Pourquoi revenir en arrière ?
Imagine un code secret de deux symboles, chacun A ou B. Tu choisis A en première
position, puis A en seconde : AA est une solution. Pour obtenir AB, tu dois
**retirer le dernier A** puis essayer B. Quand les possibilités après le premier A
sont épuisées, tu retires aussi ce premier A et explores les codes commençant par B.
L'ordre obtenu est AA, AB, BA, BB.

Le backtracking explore un arbre de décisions en profondeur. Une branche décrit
un choix ; le chemin depuis la racine décrit une solution partielle. Une feuille
peut être une solution complète ou un échec. Il sert notamment à énumérer des
combinaisons, résoudre des contraintes, chercher un chemin ou placer des pièces.

### Exemple illustré — une branche n’est qu’un essai

```text
départ
├─ choisir A → impasse
└─ choisir B → solution
```

Après l’impasse A, il faut revenir à l’état du départ pour pouvoir essayer B. Ce
retour est le cœur du backtracking.


## 2. Le schéma choisir → explorer → annuler
| Étape | Exemple | Rôle |
|---|---|---|
| Choisir | `chemin.append(choix)` | Modifier l’état de la solution partielle |
| Explorer | `visiter(...)` | Essayer toutes les suites compatibles |
| Annuler | `chemin.pop()` | Revenir exactement à l’état antérieur |

L'invariant fondamental est : **au retour d'une branche, l'état partagé est
identique à celui qu'il avait avant cette branche**. Sinon les branches suivantes
héritent de décisions qui ne leur appartiennent pas. Ce retour ne signifie pas
qu'une erreur Python s'est produite ; c'est une étape normale de l'algorithme.

### Exemple illustré — l’état avance puis revient

```text
[] → choisir A → [A] → choisir B → [A,B]
                         annuler B ← [A]
```

`append` réalise le choix ; l’appel récursif explore ; `pop` restaure exactement
l’état précédent.




*Après chaque branche, l’annulation doit restaurer exactement le chemin du parent.*

## 3. Cas de base et copie de la solution
Quand le chemin a la longueur souhaitée, on enregistre une solution et on retourne.
`resultats.append(chemin)` garde une référence vers la même liste mutable.
Après les `pop`, les résultats pourraient tous devenir vides. Utilise
`chemin.copy()` pour un historique indépendant, ou un objet immuable construit
à cet instant. Si les éléments sont eux-mêmes mutables, une copie superficielle
ne copie pas leurs contenus internes.

### Exemple illustré — copier avant de continuer à modifier

```text
chemin partagé : [A, B]
copie stockée  : [A, B]
chemin après pop : [A]
copie stockée    : [A, B]  ✓
```

Sans copie, le résultat pointerait vers la même liste mutable et changerait avec
les futurs `append` et `pop`.




*Image — La sortie ne doit pas pointer vers l’état encore mutable.*


## 4. Un arbre réduit par les contraintes
**Élaguer** consiste à abandonner une branche dont aucune continuation ne peut
satisfaire le contrat. Pour des parenthèses équilibrées, un préfixe contenant
plus de fermantes que d'ouvrantes ne peut jamais devenir valide : on le rejette
immédiatement. Ne rejette pas une branche simplement parce qu'elle ne constitue
pas encore une solution complète. « Pas encore valide » et « impossible à compléter »
sont deux situations différentes.

### Exemple illustré — couper une branche impossible

Former une somme de 5 avec des valeurs positives :

```text
somme partielle 3 → continuer
somme partielle 5 → solution
somme partielle 7 → couper : ajouter encore augmenterait la somme
```

Élaguer signifie arrêter une branche dont on peut prouver qu’elle ne donnera aucune
solution valide.




*Élaguer signifie arrêter une branche dont aucune continuation ne peut réussir.*


## 5. Exhaustivité : ne rien oublier, ne rien répéter
Pour générer des combinaisons, imposer des indices croissants évite de produire
[1,2] puis [2,1]. Une permutation traite au contraire ces ordres comme différents.
Les données avec doublons demandent un traitement explicite. Les exercices de
permutations ici supposent les éléments distincts. Énonce toujours ce choix.

### Exemple illustré — chaque suite de choix apparaît une fois

| Position 1 | Position 2 | Feuille obtenue |
|:---:|:---:|---|
| A | A | AA |
| A | B | AB |
| B | A | BA |
| B | B | BB |

Chaque niveau traite une position et chaque choix est essayé une fois : aucune
feuille n’est oubliée ni répétée.


### Trois obligations de preuve

Pour vérifier un générateur par backtracking, sépare trois questions.

1. **Validité** : toute feuille ajoutée au résultat respecte le contrat. Les
   conditions de choix et le cas de base doivent suffire à le garantir.
2. **Exhaustivité** : toute solution possible correspond à une suite de choix que
   l'algorithme peut suivre. Un élagage est sûr seulement si aucune continuation du
   préfixe rejeté ne peut réussir.
3. **Unicité** : deux chemins différents ne doivent pas représenter la même solution,
   sauf si les doublons font explicitement partie du contrat. Des indices croissants,
   un ordre canonique ou un saut de doublons peuvent assurer cette propriété.

La restauration de l'état constitue une quatrième vérification locale : après
chaque appel, toutes les mutations effectuées avant cet appel doivent être annulées
dans l'ordre inverse. Si plusieurs structures sont modifiées, elles doivent toutes
être restaurées, même lorsqu'une solution est trouvée ou qu'une branche est coupée.


## 6. Complexité : la sortie coûte aussi
Avec b choix à chacun de d niveaux, l'arbre peut avoir O(b^d) nœuds (pour b≥2).
Produire b^d chaînes de longueur d exige déjà O(d·b^d) caractères : aucun algorithme
ne peut écrire toutes ces chaînes en O(d). Un chemin et une pile de profondeur d
prennent O(d) espace de travail ; conserver toutes les solutions prend O(d·b^d).
L'élagage réduit les branches parcourues, mais ne garantit pas un coût polynomial.
Pour n éléments distincts, il existe n! permutations : leur matérialisation
demande au minimum un travail proportionnel à n·n!. Chercher une seule solution peut permettre un arrêt anticipé.

### Exemple illustré — produire les réponses prend du temps

| Longueur `n` | Mots binaires produits | Caractères à écrire |
|---:|---:|---:|
| 2 | 4 | 8 |
| 3 | 8 | 24 |
| 4 | 16 | 64 |

Même avec un parcours parfait, écrire `2ⁿ` réponses de longueur `n` demande déjà
un travail proportionnel à `n × 2ⁿ`.


## 7. Backtracking, DFS et mémoïsation
DFS décrit un ordre de parcours en profondeur. Le backtracking ajoute généralement
un état de décision et son annulation. La mémoïsation stocke des résultats de
sous-problèmes équivalents ; elle exige que la clé encode tout l'état pertinent.
Un cache indexé seulement par la profondeur serait faux si les choix déjà utilisés
changent les continuations possibles.

### Exemple illustré — trois parcours, trois intentions

| Outil | But principal | Retour en arrière |
|---|---|---|
| DFS | visiter les sommets accessibles | pour poursuivre le parcours |
| backtracking | construire toutes les solutions | avec restauration de l’état |
| mémoïsation | éviter un sous-calcul répété | résultat conservé dans un cache |

Ils peuvent tous utiliser la récursion, mais la raison de leurs appels n’est pas la
même.


## Un premier arbre entièrement tracé

Prédis où se produisent les pop et observe que le chemin redevient vide après la recherche.


In [1]:
def codes_binaires(taille, trace=False):
    if taille < 0:
        raise ValueError("taille négative")
    chemin, resultats = [], []
    def visiter():
        if len(chemin) == taille:
            resultats.append("".join(chemin))
            if trace:
                print("solution", resultats[-1])
            return
        for symbole in "AB":
            chemin.append(symbole)
            if trace:
                print("choisir", "".join(chemin))
            visiter()
            chemin.pop()
            if trace:
                print("annuler ->", "".join(chemin))
    visiter()
    assert chemin == []
    return resultats
assert codes_binaires(2, True) == ["AA","AB","BA","BB"]
assert codes_binaires(0) == [""]


choisir A
choisir AA
solution AA
annuler -> A
choisir AB
solution AB
annuler -> A
annuler -> 
choisir B
choisir BA
solution BA
annuler -> B
choisir BB
solution BB
annuler -> B
annuler -> 


### Trace visuelle de `codes_binaires(2)`

| Action | Chemin courant | Résultats déjà copiés |
|---|---|---|
| choisir A | `[A]` | `[]` |
| choisir A | `[A, A]` | `[]` |
| enregistrer puis annuler | `[A]` | `[AA]` |
| choisir B | `[A, B]` | `[AA]` |
| enregistrer puis annuler deux fois | `[]` | `[AA, AB]` |
| choisir B puis explorer A et B | `[]` à la fin | `[AA, AB, BA, BB]` |

Les résultats reçoivent une copie. Le même objet `chemin` continue ensuite à être
modifié pour explorer les autres branches.

Taille 0 : il existe une chaîne vide, donc la réponse est [""], pas []. Le coût de production est O(d·2^d) pour d≥1, avec O(d) espace de travail et O(d·2^d) sortie.


## Sous-ensembles : inclure ou exclure

Pour chaque indice, les deux choix sont exclusifs et exhaustifs.


In [2]:
def sous_ensembles(valeurs):
    chemin, resultats = [], []
    def visiter(i):
        if i == len(valeurs):
            resultats.append(chemin.copy())
            return
        visiter(i+1)  # ne pas prendre valeurs[i]
        chemin.append(valeurs[i])
        visiter(i+1)  # prendre valeurs[i]
        chemin.pop()
    visiter(0)
    return resultats
print(sous_ensembles([1,2]))
assert sous_ensembles([1,2]) == [[],[2],[1],[1,2]]
assert sous_ensembles([]) == [[]]


[[], [2], [1], [1, 2]]


Pour n positions : 2^n sous-ensembles, y compris le vide. O(n·2^n) temps et sortie dans le pire cas, O(n) espace de travail. Avec des valeurs répétées, des sous-ensembles de mêmes valeurs peuvent apparaître : la fonction énumère des choix de positions.


## Exercices différenciés

Essaie chaque exercice avant de consulter le corrigé. Pour lancer les tests d’un exercice, remplace son `pass`, puis enlève le `#` devant les appels de test. Les cellules incomplètes restent exécutables.


### Palier A — Mots sur un alphabet

Renvoie tous les mots de longueur n sur un alphabet fourni comme chaîne de caractères distincts (par exemple "abc"). Alphabet vide et n>0 donnent []; n=0 donne [""]. Refuse n<0.

<details><summary>Indice progressif</summary>

Adapte le premier exemple ; construis la chaîne à la feuille.

</details>


In [3]:
def mots(alphabet, n):
    pass


In [4]:
# assert mots("ab",2) == ["aa","ab","ba","bb"]
# assert mots("",2) == []
# assert mots("ab",0) == [""]


### Palier B — Combinaisons de k indices

Reçois n≥0 et k≥0. Renvoie les combinaisons de k indices parmi 0,…,n−1, sans changement d’ordre ni répétition. Pour k>n, renvoie [].

<details><summary>Indice progressif</summary>

Conserve le prochain indice autorisé. Si trop peu d’indices restent pour finir, coupe la branche.

</details>


In [5]:
def combinaisons(n, k):
    pass


In [6]:
# assert combinaisons(4,2) == [[0,1],[0,2],[0,3],[1,2],[1,3],[2,3]]
# assert combinaisons(2,0) == [[]]
# assert combinaisons(2,3) == []


### Palier C — Parenthèses équilibrées

Renvoie toutes les chaînes de n paires de parenthèses correctement imbriquées, n≥0. Pour n=0, [""].

<details><summary>Indice progressif</summary>

Ajoute une ouvrante si ouvertes<n ; ajoute une fermante si fermees<ouvertes.

</details>


In [7]:
def parentheses(n):
    pass


In [8]:
# assert parentheses(0) == [""]
# assert parentheses(1) == ["()"]
# assert parentheses(3) == ["((()))","(()())","(())()","()(())","()()()"]


## Questions de compréhension

1. Quelle différence y a-t-il entre un préfixe incomplet et un préfixe impossible ?
2. Comment justifier validité, exhaustivité et unicité des résultats ?
3. Pourquoi faut-il copier `chemin` à une feuille ?
4. Pourquoi une optimisation ne peut-elle pas rendre bon marché une sortie de n! objets ?


## À retenir et vérifier

Avant de récursiver : que représente l’état ? Quels choix restent ? Quand la solution est-elle complète ? Après la récursion : ai-je annulé toutes les modifications ? N’utilise Sudoku ou N-Queens qu’après avoir maîtrisé ces questions sur de petits arbres.

**Autoévaluation :** peux-tu expliquer l'état des variables sur un petit exemple,
justifier la condition d'arrêt et donner un cas limite ? Peux-tu distinguer
l'espace de travail de l'espace occupé par les résultats ?

**Débrief en binôme :** présente une première solution correcte, puis un compromis
(vitesse, mémoire, lisibilité). Une optimisation doit préserver le contrat.


**Étape suivante.** Le chapitre 7 généralise ces parcours : dans un graphe, plusieurs chemins peuvent se rejoindre et former des cycles.
